# ML Toolkit - Getting Started

This notebook demonstrates the core features of the ML Toolkit.

## Setup

In [ ]:
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, '../src')

from ml_toolkit.data import DataLoader, BasePreprocessor, CustomDataset
from ml_toolkit.training import Trainer, Evaluator
from ml_toolkit.config import ConfigManager, TrainingConfig
from ml_toolkit.logging import ExperimentLogger
from ml_toolkit.visualization import plot_training_history, setup_plotting_style

# Setup plotting
setup_plotting_style()
%matplotlib inline

## 1. Data Loading and Preprocessing

In [ ]:
# Generate synthetic data
n_samples = 1000
input_size = 20
output_size = 3

X_train = torch.randn(n_samples, input_size)
y_train = torch.randint(0, output_size, (n_samples,))

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")

In [ ]:
# Create dataset
train_dataset = CustomDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches: {len(train_loader)}")

## 2. Model Definition

In [ ]:
# Define a simple neural network
model = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, output_size),
)

print(model)

## 3. Configuration Management

In [ ]:
# Create training configuration
config = TrainingConfig(
    epochs=10,
    learning_rate=0.001,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(config.model_dump())

## 4. Training

In [ ]:
# Setup training components
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

# Initialize experiment logger
logger = ExperimentLogger('notebook_experiment', backend='local')
logger.log_params(config.model_dump())

# Create trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=config.device
)

In [ ]:
# Train the model
history = trainer.train(train_loader, epochs=config.epochs)

## 5. Visualization

In [ ]:
# Plot training history
plot_training_history(history)

## 6. Evaluation

In [ ]:
# Evaluate on training data (just for demonstration)
evaluator = Evaluator(model, criterion, config.device)
train_loss = evaluator.evaluate(train_loader)

print(f"Final training loss: {train_loss:.4f}")

In [ ]:
# Get predictions
predictions = evaluator.predict(train_loader)
print(f"Number of predictions: {len(predictions)}")
print(f"Prediction shape: {predictions[0].shape}")

## 7. Save Results

In [ ]:
# Log final metrics and finish
logger.log_metrics({
    'final_train_loss': history['train_loss'][-1],
    'min_train_loss': min(history['train_loss'])
})

logger.finish()
print("Experiment completed and logged!")